# Algorithm 8.1 — Planet state vector at an epoch

**Goal:** given a planet and a date/time, return its **heliocentric** orbital elements and state vector $(\mathbf{r},\mathbf{v})$ — the ephemeris that interplanetary mission design starts from.

**Code (answer key):** [`planet_elements_and_sv`](../curtis/algorithms/alg_8_1_planet_elements_and_sv.py) · **Book:** §8.10, Algorithm 8.1 · **Worked example:** 8.7

It reuses everything: a date → Julian date (like 5.3), Kepler's equation (`kepler_E`, 3.1), and elements → state (`sv_from_coe`, 4.2). The new ingredient is a **table of how each orbital element drifts with time**.

## Read first

| Symbol | Meaning |
|---|---|
| `planet_id` | 1=Mercury … 8=Neptune, 9=Pluto |
| $t_0$ | Julian centuries since J2000 |
| $a,e,i,\Omega,\varpi,L$ | semimajor axis, eccentricity, inclination, node, **longitude of perihelion**, **mean longitude** |
| $\varpi=\Omega+\omega$ | longitude of perihelion |
| $L=\varpi+M$ | mean longitude |
| output | `coe` (11 elements), $\mathbf{r},\mathbf{v}$ (heliocentric), `jd` |

Frame: **heliocentric ecliptic** (Sun at the focus, ecliptic as the reference plane), with $\mu=\mu_{\odot}$.

## The picture (an ephemeris from a drift table)

Planetary orbits are nearly fixed, but they *precess and drift* slowly. Table 8.1 captures this with, for each planet, a **J2000 value and a rate per century** for every element:
$$\text{element}(t_0)=\text{element}_{\text{J2000}}+\dot{\text{element}}\cdot t_0,\qquad t_0=\frac{\text{JD}-2451545}{36525}.$$

So the algorithm is:

1. **Date → time** : Julian date, then centuries past J2000.
2. **Drift the elements** to that epoch with the table.
3. **Anomalies** : the table gives the *mean longitude* $L$; convert to mean anomaly $M=L-\varpi$, solve Kepler's equation for $E$ (`kepler_E`), then true anomaly $\theta$.
4. **State vector** : feed the elements to `sv_from_coe` (Algorithm 4.2).

The figure shows the result for the inner planets at the example epoch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, "..")
from curtis.algorithms.julian_day import J0
from curtis.algorithms.alg_3_1_kepler_E import kepler_E
from curtis.algorithms.alg_4_2_sv_from_coe import sv_from_coe
# Provided: Table 8.1 (with AU->km, arcsec->deg conversions) and the 0-360 reducer.
from curtis.algorithms.alg_8_1_planet_elements_and_sv import planetary_elements, zero_to_360
# Reference, for the picture only:
from curtis.algorithms.alg_8_1_planet_elements_and_sv import planet_elements_and_sv as _pe_ref

In [ ]:
# The inner solar system at the epoch of Example 8.7 (2003-08-27 12:00 UT).
mu_sun = 1.327124e11
AU = 149597871.0
names = {1: "Mercury", 2: "Venus", 3: "Earth", 4: "Mars"}
cols  = {1: "#888780", 2: "#D85A30", 3: "#3B8BD4", 4: "#D4537E"}

fig, ax = plt.subplots(figsize=(6.6, 6.6))
ax.plot(0, 0, 'o', color='#F2A623', ms=14)                     # Sun
ax.annotate('Sun', (0, 0), textcoords='offset points', xytext=(8, -12), color='#B05535')
for pid in (1, 2, 3, 4):
    coe, r, v, jd = _pe_ref(pid, 2003, 8, 27, 12, 0, 0, mu=mu_sun)
    h, e, RA, incl, w = coe[0], coe[1], coe[2], coe[3], coe[4]
    deg = np.pi/180
    th = np.linspace(0, 2*np.pi, 400)
    orbit = np.array([sv_from_coe([h, e, RA*deg, incl*deg, w*deg, t], mu_sun)[0] for t in th])
    ax.plot(orbit[:,0]/AU, orbit[:,1]/AU, color=cols[pid], lw=1.2)
    ax.plot(r[0]/AU, r[1]/AU, 'o', color=cols[pid], ms=7)
    ax.annotate(names[pid], (r[0]/AU, r[1]/AU), textcoords='offset points',
                xytext=(7, 4), color=cols[pid], fontsize=10)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Inner solar system at 2003-08-27 (Algorithm 8.1)\n(positions projected onto the ecliptic, in AU)')
plt.show()

## See it — where each planet is on its date

Each coloured ellipse is a planet's orbit; the dot is where Algorithm 8.1 puts it at the example epoch (projected onto the ecliptic, scaled in AU, Sun at the focus). Earth (blue) is the one Example 8.7 computes in full. Run 8.1 for any planet and date and you get the dot — the starting point for planning a transfer between two of them (that is Algorithm 8.2).

## Where these equations come from

### Secular variation (Table 8.1)
Over decades, perturbations make each classical element creep almost linearly. Table 8.1 lists, per planet, the J2000 value and a per-century rate; the element at epoch is $\text{val}_{\text{J2000}}+\text{rate}\cdot t_0$. (`planetary_elements` returns these with $a$ converted from AU to km and the angular rates from arcsec/century to deg/century, so everything is in km and degrees.)

### Heliocentric angle conventions
Tables of planets use **longitude of perihelion** $\varpi=\Omega+\omega$ and **mean longitude** $L=\varpi+M$ (angles measured continuously around the ecliptic, convenient because $\Omega$ and $\omega$ are individually ill-defined for nearly-circular, nearly-planar orbits). To get the usual elements you invert these: $\omega=\varpi-\Omega$ and $M=L-\varpi$.

### Anomaly chain
The table gives $M$ (via $L$), but `sv_from_coe` needs the **true** anomaly $\theta$. So solve Kepler's equation $M=E-e\sin E$ for $E$ (`kepler_E`, Algorithm 3.1), then
$$\theta=2\tan^{-1}\!\Big(\sqrt{\tfrac{1+e}{1-e}}\,\tan\tfrac{E}{2}\Big)\qquad(3.10).$$

### State vector
With $h=\sqrt{\mu a(1-e^2)}$ and all six elements known, Algorithm 4.2 (`sv_from_coe`) returns the heliocentric $\mathbf{r},\mathbf{v}$.

## Step by step (in code order)

1. **Julian date:** `j0 = J0(year, month, day)`; `ut = (hour + minute/60 + second/3600)/24`; `jd = j0 + ut`.
2. **Centuries since J2000:** `t0 = (jd - 2451545)/36525`.
3. **Table 8.1 (provided):** `J2000_coe, rates = planetary_elements(planet_id)`.
4. **Drift to epoch:** `elements = J2000_coe + rates*t0`.
5. `a, e = elements[0], elements[1]`; `h = sqrt(mu*a*(1-e**2))`.
6. **Reduce angles** with `zero_to_360`: `incl`, `RA`, `w_hat`, `L`; then `w = zero_to_360(w_hat - RA)`, `M = zero_to_360(L - w_hat)`.
7. **Eccentric anomaly:** `E = kepler_E(e, M*deg)`.
8. **True anomaly:** `TA = zero_to_360(2*atan(sqrt((1+e)/(1-e))*tan(E/2))/deg)`.
9. `coe = [h, e, RA, incl, w, TA, a, w_hat, L, M, E/deg]`.
10. **State vector:** `r, v = sv_from_coe([h, e, RA*deg, incl*deg, w*deg, TA*deg], mu)`.

**↓ Now type the implementation below.** `planetary_elements`, `zero_to_360`, `J0`, `kepler_E`, `sv_from_coe` are all provided/imported. Answer key linked above; peek only after you try.

In [ ]:
def planet_elements_and_sv(planet_id, year, month, day, hour, minute, second, mu):
    """Heliocentric orbital elements and state vector of a planet at an epoch."""
    deg = np.pi/180

    # 1. Julian date:
    #     j0 = J0(year, month, day); ut = (hour + minute/60 + second/3600)/24; jd = j0 + ut
    # 2. t0 = (jd - 2451545)/36525
    # 3. J2000_coe, rates = planetary_elements(planet_id)           # provided (Table 8.1)
    # 4. elements = J2000_coe + rates*t0
    # 5. a = elements[0]; e = elements[1]; h = sqrt(mu*a*(1 - e**2))
    # 6. Reduce angles (deg):
    #     incl  = elements[2]
    #     RA    = zero_to_360(elements[3])
    #     w_hat = zero_to_360(elements[4])
    #     L     = zero_to_360(elements[5])
    #     w = zero_to_360(w_hat - RA)        # argument of perihelion
    #     M = zero_to_360(L - w_hat)          # mean anomaly
    # 7. E = kepler_E(e, M*deg)                                      (Algorithm 3.1)
    # 8. TA = zero_to_360(2*np.arctan(np.sqrt((1+e)/(1-e))*np.tan(E/2))/deg)   (Eq 3.10)
    # 9. coe = [h, e, RA, incl, w, TA, a, w_hat, L, M, E/deg]
    # 10. r, v = sv_from_coe([h, e, RA*deg, incl*deg, w*deg, TA*deg], mu)      (Algorithm 4.2)
    # return coe, r, v, jd
    raise NotImplementedError("fill in steps 1-10, then delete this line")

## Verify — Example 8.7

**Input:** Earth (`planet_id = 3`), 2003-08-27, 12:00:00 UT, $\mu_\odot=1.327124\times10^{11}$ km³/s².

**Expected:** JD $=2452879.0$; $e=0.0167088$, $a=1.49598\times10^8$ km, $M=232.308°$, $E=231.558°$, $\theta=230.812°$; $\mathbf{r}=[1.35589\times10^8,\,-6.68029\times10^7,\,286.909]$ km, $\mathbf{v}=[12.6804,\,26.61,\,-2.13\times10^{-4}]$ km/s.

Run once your function is typed.

In [ ]:
mu_sun = 1.327124e11
coe, r, v, jd = planet_elements_and_sv(3, 2003, 8, 27, 12, 0, 0, mu_sun)
h, e, RA, incl, w, TA, a, w_hat, L, M, E = coe

print(f"JD = {jd:.7g}   (expect 2452879.0)")
print(f"a = {a:.6g} km   e = {e:.6g}   M = {M:.6g}   E = {E:.6g}   TA = {TA:.6g}")
print(f"r (km)   = [{r[0]:.6g} {r[1]:.6g} {r[2]:.6g}]")
print(f"v (km/s) = [{v[0]:.6g} {v[1]:.6g} {v[2]:.6g}]")

assert abs(jd - 2452879.0) < 1e-3
assert abs(e - 0.0167088) < 1e-6
assert abs(a - 1.49598e8) < 1e4
assert abs(M - 232.308) < 1e-2 and abs(TA - 230.812) < 1e-2
assert np.allclose(r, [1.35589e8, -6.68029e7, 286.909], rtol=1e-4)
assert np.allclose(v, [12.6804, 26.61, -0.000212731], rtol=1e-4, atol=1e-5)
print("\nAll checks passed ✔")

## What this confirms

- A planet's position on any date comes from a **drift table** (J2000 value + rate per century) plus the same anomaly machinery as Earth-orbit problems — just heliocentric.
- The **longitude conventions** ($\varpi=\Omega+\omega$, $L=\varpi+M$) are bookkeeping you undo to recover the standard elements.
- It composes prior algorithms end to end: `J0` → drift → `kepler_E` → `sv_from_coe`.

**Next:** Algorithm 8.2 (`interplanetary`) — the book's capstone. Call this for the departure planet and the arrival planet, then connect their positions with `lambert` (5.2) to design a patched-conic transfer and read off the hyperbolic excess velocities.